# All-In-One Natural AI Notebook

This notebook is self-contained for Colab.

It does all major steps in one place:
1. Mount Google Drive
2. Load DDXPlus dataset from Drive
3. Build natural-language CSVs without label leakage
4. Train ClinicalBERT classifier
5. Rebuild FAISS index from natural text only
6. Export final artifacts to Drive

You only need to change the paths in the configuration cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas tqdm datasets transformers torch faiss-cpu sentencepiece accelerate scikit-learn seaborn matplotlib

In [ ]:
from pathlib import Path

# Update these 2 paths only
DDX_DIR = Path('/content/drive/MyDrive/DDX/raw/ddxplus_hf')
EXPORT_ROOT = Path('/content/drive/MyDrive/DDX/exports_natural_ai')
EVIDENCE_MAP_PATH = DDX_DIR / 'release_evidences.json'
CONDITIONS_MAP_PATH = DDX_DIR / 'release_conditions.json'

# Choose exactly one profile: small, mid, large, very_large, full_dataset
DATASET_PROFILE = 'mid'
SEED = 42

DATASET_PROFILES = {
    'small': {
        'description': 'Fast smoke test',
        'limits': {'train': 2_000, 'validate': 500, 'test': 500},
        'epochs': 1,
    },
    'mid': {
        'description': 'Balanced Colab run',
        'limits': {'train': 10_000, 'validate': 2_000, 'test': 2_000},
        'epochs': 2,
    },
    'large': {
        'description': 'Heavier training run',
        'limits': {'train': 50_000, 'validate': 10_000, 'test': 10_000},
        'epochs': 2,
    },
    'very_large': {
        'description': 'Long Colab run',
        'limits': {'train': 200_000, 'validate': 30_000, 'test': 30_000},
        'epochs': 2,
    },
    'full_dataset': {
        'description': 'Use every available row',
        'limits': {'train': None, 'validate': None, 'test': None},
        'epochs': 3,
    },
}

assert DATASET_PROFILE in DATASET_PROFILES, (
    f'Unknown DATASET_PROFILE={DATASET_PROFILE!r}. '
    f'Choose one of: {list(DATASET_PROFILES)}'
)

PROFILE_CONFIG = DATASET_PROFILES[DATASET_PROFILE]
PROFILE_SLUG = DATASET_PROFILE
PROFILE_EXPORT_DIR = EXPORT_ROOT / PROFILE_SLUG
WORK_DIR = Path(f'/content/natural_ai_work_{PROFILE_SLUG}')
PROCESSED_DIR = WORK_DIR / 'processed_ddxplus'
CLASSIFIER_DIR = WORK_DIR / 'clinicalbert_classifier_natural'
FAISS_DIR = WORK_DIR / 'faiss_data_natural'

PROFILE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CLASSIFIER_DIR.mkdir(parents=True, exist_ok=True)
FAISS_DIR.mkdir(parents=True, exist_ok=True)

assert DDX_DIR.exists(), f'DDX dataset path not found: {DDX_DIR}'
assert EVIDENCE_MAP_PATH.exists(), f'Evidence mapping file not found: {EVIDENCE_MAP_PATH}'
print('DDX_DIR =', DDX_DIR)
print('EVIDENCE_MAP_PATH =', EVIDENCE_MAP_PATH)
print('CONDITIONS_MAP_PATH =', CONDITIONS_MAP_PATH)
print('EXPORT_ROOT =', EXPORT_ROOT)
print('PROFILE_EXPORT_DIR =', PROFILE_EXPORT_DIR)
print('WORK_DIR =', WORK_DIR)
print('DATASET_PROFILE =', DATASET_PROFILE)
print('PROFILE_CONFIG =', PROFILE_CONFIG)



## Build Natural-Language CSVs

Important: this step removes label leakage.

The generated `combined_text` contains only:
- patient demographics
- presenting symptoms

It does **not** include diagnosis labels or differential diagnosis inside the text.

In [ ]:
import ast
import json
import pandas as pd
from datasets import load_from_disk

dataset = load_from_disk(str(DDX_DIR))
with open(EVIDENCE_MAP_PATH, 'r', encoding='utf-8') as fh:
    evidence_catalog = json.load(fh)
print('Available splits:', list(dataset.keys()))
print('Loaded evidences:', len(evidence_catalog))

class EvidenceMapper:
    def __init__(self, catalog):
        self.catalog = catalog

    @staticmethod
    def _normalize_text(value):
        text = str(value or '').strip()
        return ' '.join(text.replace('_', ' ').split())

    def _question_text(self, meta):
        for key in ('question_en', 'question', 'label', 'description'):
            value = meta.get(key)
            if isinstance(value, str) and value.strip():
                return self._normalize_text(value)
        return ''

    def _value_text(self, meta, value_code):
        value_meaning = meta.get('value_meaning') or {}
        if isinstance(value_meaning, dict):
            raw_value = value_meaning.get(value_code)
            if isinstance(raw_value, dict):
                for key in ('en', 'english', 'label', 'name'):
                    candidate = raw_value.get(key)
                    if isinstance(candidate, str) and candidate.strip():
                        return self._normalize_text(candidate)
            elif isinstance(raw_value, str) and raw_value.strip():
                return self._normalize_text(raw_value)

        possible_values = meta.get('possible-values') or meta.get('possible_values') or []
        if isinstance(possible_values, dict):
            raw_value = possible_values.get(value_code)
            if isinstance(raw_value, str) and raw_value.strip():
                return self._normalize_text(raw_value)
        return self._normalize_text(value_code)

    def get_text(self, code):
        raw_code = str(code).strip()
        if not raw_code:
            return ''

        if '_@_' in raw_code:
            base_code, value_code = raw_code.split('_@_', 1)
        else:
            base_code, value_code = raw_code, None

        meta = self.catalog.get(base_code) or {}
        question_text = self._question_text(meta)
        if not question_text:
            fallback = base_code.replace('_', ' ').replace('-', ' ')
            fallback = fallback.replace('E ', '').replace('e ', '')
            question_text = ' '.join(fallback.split())

        if value_code is None:
            return question_text

        value_text = self._value_text(meta, value_code)
        if value_text:
            return f'{question_text}: {value_text}'
        return question_text

class DDXNaturalPreprocessor:
    def __init__(self):
        self.evidence_mapper = EvidenceMapper(evidence_catalog)

    def parse_evidences(self, evidences_data):
        if evidences_data is None:
            return []
        if isinstance(evidences_data, list):
            symptoms = []
            for item in evidences_data:
                if isinstance(item, str) and item.strip():
                    symptoms.append(self.evidence_mapper.get_text(item))
                elif isinstance(item, dict):
                    for key in ('name', 'code', 'symptom'):
                        if key in item:
                            symptoms.append(self.evidence_mapper.get_text(item[key]))
                            break
            return symptoms
        if isinstance(evidences_data, dict):
            return [
                self.evidence_mapper.get_text(code)
                for code, value in evidences_data.items()
                if value in [1, 'Y', True, 'yes', '1', 1.0]
            ]
        if isinstance(evidences_data, str) and evidences_data.strip():
            for parser in (ast.literal_eval, json.loads):
                try:
                    parsed = parser(evidences_data)
                    return self.parse_evidences(parsed)
                except Exception:
                    pass
            return [evidences_data.strip()]
        return []

    @staticmethod
    def build_combined_text(age, sex, symptoms_text):
        parts = []
        if age not in (None, '', 'Unknown') and sex not in (None, '', 'Unknown'):
            parts.append(f'Patient: {age} year old {sex}')
        if symptoms_text and symptoms_text != 'None reported':
            parts.append(f'Presenting symptoms: {symptoms_text}')
        return '. '.join(parts) if parts else 'No information available'

    def process_split(self, split_dataset, split_name):
        rows = []
        for idx, row in enumerate(split_dataset):
            evidences = []
            for col_name in ('EVIDENCES', 'evidences', 'symptoms', 'evidence'):
                if col_name in row and row[col_name] is not None:
                    evidences = self.parse_evidences(row[col_name])
                    if evidences:
                        break

            pathology = 'Unknown'
            for col_name in ('PATHOLOGY', 'pathology', 'diagnosis', 'condition'):
                if col_name in row and row[col_name] is not None:
                    pathology = str(row[col_name]).strip()
                    break

            age = 'Unknown'
            for col_name in ('AGE', 'age'):
                if col_name in row and not pd.isna(row[col_name]):
                    age = int(row[col_name])
                    break

            sex = 'Unknown'
            for col_name in ('SEX', 'sex', 'gender'):
                if col_name in row and row[col_name] is not None:
                    sex = str(row[col_name]).strip()
                    break

            symptoms_text = ', '.join(evidences) if evidences else 'None reported'
            rows.append({
                'patient_id': f'{split_name}_{idx}',
                'age': age,
                'sex': sex,
                'symptoms_text': symptoms_text,
                'pathology': pathology,
                'combined_text': self.build_combined_text(age, sex, symptoms_text),
            })
        return pd.DataFrame(rows)

def resolve_split(source_split):
    return 'validate' if source_split == 'validation' else source_split

def sample_split(split_dataset, split_name):
    limit = PROFILE_CONFIG['limits'].get(split_name)
    total_rows = len(split_dataset)
    if limit is None or limit >= total_rows:
        print(f'Using full {split_name} split: {total_rows:,} rows')
        return split_dataset, total_rows

    sampled = split_dataset.shuffle(seed=SEED).select(range(limit))
    print(f'Sampled {split_name}: {limit:,} / {total_rows:,} rows')
    return sampled, total_rows

preprocessor = DDXNaturalPreprocessor()
written = {}
source_splits = ('train', 'validate', 'validation', 'test')
print(f'Building natural CSVs with profile: {DATASET_PROFILE}')
print('Profile limits:', PROFILE_CONFIG['limits'])

for source_split in source_splits:
    if source_split not in dataset:
        continue
    split_name = resolve_split(source_split)
    sampled_split, total_rows = sample_split(dataset[source_split], split_name)
    out_df = preprocessor.process_split(sampled_split, split_name)
    out_path = PROCESSED_DIR / f'{split_name}_natural.csv'
    out_df.to_csv(out_path, index=False, encoding='utf-8')
    written[split_name] = {
        'rows': len(out_df),
        'source_rows': total_rows,
        'profile': DATASET_PROFILE,
        'path': str(out_path),
    }

print(json.dumps(written, indent=2))



## Train ClinicalBERT Classifier

In [ ]:
import csv
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
TEXT_COLUMN = 'combined_text'
LABEL_COLUMN = 'pathology'
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = PROFILE_CONFIG['epochs']
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print({'profile': DATASET_PROFILE, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'device': DEVICE})

def read_csv_rows(path, text_column, label_column):
    rows = []
    with open(path, 'r', encoding='utf-8-sig', newline='') as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            text = (row.get(text_column) or '').strip()
            label = (row.get(label_column) or '').strip()
            if text and label:
                rows.append({'text': text, 'label': label})
    return rows

train_rows = read_csv_rows(PROCESSED_DIR / 'train_natural.csv', TEXT_COLUMN, LABEL_COLUMN)
val_rows = read_csv_rows(PROCESSED_DIR / 'validate_natural.csv', TEXT_COLUMN, LABEL_COLUMN)
test_rows = read_csv_rows(PROCESSED_DIR / 'test_natural.csv', TEXT_COLUMN, LABEL_COLUMN)

print({'train_rows': len(train_rows), 'val_rows': len(val_rows), 'test_rows': len(test_rows)})

labels = sorted({row['label'] for row in train_rows})
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DiagnosisDataset(Dataset):
    def __init__(self, rows, tokenizer, label2id, max_length):
        self.rows = rows
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        encoded = self.tokenizer(
            row['text'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item['labels'] = torch.tensor(self.label2id[row['label']], dtype=torch.long)
        item['raw_text'] = row['text']
        return item

def collate_batch(batch):
    keys = ('input_ids', 'attention_mask', 'labels')
    collated = {key: torch.stack([item[key] for item in batch]) for key in keys}
    if 'token_type_ids' in batch[0]:
        collated['token_type_ids'] = torch.stack([item['token_type_ids'] for item in batch])
    collated['raw_text'] = [item['raw_text'] for item in batch]
    return collated

def move_to_device(batch, device):
    moved = {}
    for key, value in batch.items():
        moved[key] = value.to(device) if isinstance(value, torch.Tensor) else value
    return moved

def accuracy_score(y_true, y_pred):
    return sum(int(t == p) for t, p in zip(y_true, y_pred)) / len(y_true) if y_true else 0.0

def build_confusion_matrix(y_true, y_pred, labels):
    idx_map = {label: i for i, label in enumerate(labels)}
    matrix = [[0 for _ in labels] for _ in labels]
    for true_label, pred_label in zip(y_true, y_pred):
        matrix[idx_map[true_label]][idx_map[pred_label]] += 1
    return matrix

def build_classification_report(y_true, y_pred, labels, id2label):
    supports = Counter(y_true)
    rows = []
    macro_precision = 0.0
    macro_recall = 0.0
    macro_f1 = 0.0
    weighted_precision = 0.0
    weighted_recall = 0.0
    weighted_f1 = 0.0
    for label in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)
        support = supports.get(label, 0)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        rows.append({'label': id2label[label], 'precision': precision, 'recall': recall, 'f1_score': f1, 'support': support})
        macro_precision += precision
        macro_recall += recall
        macro_f1 += f1
        weighted_precision += precision * support
        weighted_recall += recall * support
        weighted_f1 += f1 * support
    total = len(y_true) if y_true else 1
    class_count = len(labels) if labels else 1
    averages = {
        'macro_precision': macro_precision / class_count,
        'macro_recall': macro_recall / class_count,
        'macro_f1': macro_f1 / class_count,
        'weighted_precision': weighted_precision / total,
        'weighted_recall': weighted_recall / total,
        'weighted_f1': weighted_f1 / total,
    }
    return rows, averages

def evaluate_model(model, dataloader, device, id2label):
    model.eval()
    losses = []
    y_true = []
    y_pred = []
    prediction_rows = []
    with torch.no_grad():
        for batch in dataloader:
            batch = move_to_device(batch, device)
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                labels=batch['labels'],
                token_type_ids=batch.get('token_type_ids'),
            )
            losses.append(float(outputs.loss.item()))
            preds = torch.argmax(outputs.logits, dim=1)
            true_ids = batch['labels'].detach().cpu().tolist()
            pred_ids = preds.detach().cpu().tolist()
            y_true.extend(true_ids)
            y_pred.extend(pred_ids)
            for text, true_id, pred_id in zip(batch['raw_text'], true_ids, pred_ids):
                prediction_rows.append({
                    'text': text,
                    'true_label': id2label[true_id],
                    'predicted_label': id2label[pred_id],
                    'correct': str(true_id == pred_id),
                })
    labels = sorted(set(y_true) | set(y_pred))
    confusion = build_confusion_matrix(y_true, y_pred, labels)
    report_rows, averages = build_classification_report(y_true, y_pred, labels, id2label)
    return {
        'loss': sum(losses) / len(losses) if losses else 0.0,
        'accuracy': accuracy_score(y_true, y_pred),
        'labels': labels,
        'confusion': confusion,
        'report_rows': report_rows,
        'averages': averages,
        'prediction_rows': prediction_rows,
    }

train_dataset = DiagnosisDataset(train_rows, tokenizer, label2id, MAX_LENGTH)
val_dataset = DiagnosisDataset(val_rows, tokenizer, label2id, MAX_LENGTH)
test_dataset = DiagnosisDataset(test_rows, tokenizer, label2id, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = max(1, len(train_loader) * EPOCHS)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, math.ceil(total_steps * 0.1)),
    num_training_steps=total_steps,
)

history = []
best_metric = -1.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels'],
            token_type_ids=batch.get('token_type_ids'),
        )
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_losses.append(float(loss.item()))
    val_metrics = evaluate_model(model, val_loader, DEVICE, id2label)
    epoch_record = {
        'epoch': epoch,
        'train_loss': sum(train_losses) / len(train_losses),
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
        'val_macro_f1': val_metrics['averages']['macro_f1'],
    }
    history.append(epoch_record)
    print(epoch_record)
    if val_metrics['averages']['macro_f1'] > best_metric:
        best_metric = val_metrics['averages']['macro_f1']
        best_epoch = epoch
        model.save_pretrained(CLASSIFIER_DIR)
        tokenizer.save_pretrained(CLASSIFIER_DIR)
        with open(CLASSIFIER_DIR / 'label_map.json', 'w', encoding='utf-8') as fh:
            json.dump({'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, fh, indent=2)

best_model = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_DIR).to(DEVICE)
test_metrics = evaluate_model(best_model, test_loader, DEVICE, id2label)

with open(CLASSIFIER_DIR / 'history.json', 'w', encoding='utf-8') as fh:
    json.dump(history, fh, indent=2)
with open(CLASSIFIER_DIR / 'summary.json', 'w', encoding='utf-8') as fh:
    json.dump({
        'profile': DATASET_PROFILE,
        'best_epoch': best_epoch,
        'best_val_macro_f1': best_metric,
        'test_loss': test_metrics['loss'],
        'test_accuracy': test_metrics['accuracy'],
        'test_macro_f1': test_metrics['averages']['macro_f1'],
        'rows': {
            'train': len(train_rows),
            'validate': len(val_rows),
            'test': len(test_rows),
        },
    }, fh, indent=2)

pd.DataFrame(test_metrics['report_rows']).to_csv(CLASSIFIER_DIR / 'test_classification_report.csv', index=False, encoding='utf-8')
pd.DataFrame(test_metrics['prediction_rows']).to_csv(CLASSIFIER_DIR / 'test_predictions.csv', index=False, encoding='utf-8')
pd.DataFrame(test_metrics['confusion'], index=[id2label[idx] for idx in test_metrics['labels']], columns=[id2label[idx] for idx in test_metrics['labels']]).to_csv(CLASSIFIER_DIR / 'test_confusion_matrix.csv', encoding='utf-8')

print({
    'profile': DATASET_PROFILE,
    'best_epoch': best_epoch,
    'best_val_macro_f1': best_metric,
    'test_accuracy': test_metrics['accuracy'],
    'test_macro_f1': test_metrics['averages']['macro_f1'],
})



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm_df = pd.read_csv(CLASSIFIER_DIR / 'test_confusion_matrix.csv')
display(cm_df.head())
plt.figure(figsize=(18, 14))
sns.heatmap(cm_df.set_index('true\\pred'), cmap='Blues')
plt.title('ClinicalBERT Test Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## Rebuild FAISS From Natural Text Only

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
import faiss
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

EMBED_MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
EMBED_BATCH_SIZE = 32 if DEVICE == 'cuda' else 8
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
embed_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE)
embed_model.eval()

print({'profile': DATASET_PROFILE, 'embed_batch_size': EMBED_BATCH_SIZE, 'device': DEVICE})

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

def encode_texts(texts, batch_size):
    vectors = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        inputs = embed_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt',
        ).to(DEVICE)
        with torch.no_grad():
            outputs = embed_model(**inputs)
        embeddings = mean_pooling(outputs, inputs['attention_mask'])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        vectors.append(embeddings.cpu().numpy().astype('float32'))
    return np.vstack(vectors)

frames = []
for split_name in ('train', 'validate'):
    csv_path = PROCESSED_DIR / f'{split_name}_natural.csv'
    if csv_path.exists():
        frame = pd.read_csv(csv_path)
        frame['source_split'] = split_name
        frames.append(frame)

assert frames, 'No train/validate natural CSVs found for FAISS build.'
combined_df = pd.concat(frames, ignore_index=True)
combined_df['combined_text'] = combined_df['combined_text'].fillna('No information available')

def looks_like_encoded_ddx_text(text):
    if not isinstance(text, str):
        return False
    marker = 'Presenting symptoms:'
    if marker in text:
        text = text.split(marker, 1)[1].strip()
    return bool(text) and all(ch.isdigit() or ch in ' ,.@Vv_-' for ch in text)

encoded_examples = [text for text in combined_df['combined_text'].head(20).tolist() if looks_like_encoded_ddx_text(text)]
assert not encoded_examples, (
    'combined_text is still encoded instead of natural clinical text. '
    'Fix evidence mapping before exporting FAISS. Example: ' + encoded_examples[0]
)
text_rows = combined_df['combined_text'].tolist()

print({'faiss_rows': len(text_rows), 'faiss_splits': sorted(combined_df['source_split'].unique().tolist())})
embeddings = encode_texts(text_rows, EMBED_BATCH_SIZE)
faiss.normalize_L2(embeddings)
dimension = embeddings.shape[1]
nlist = min(100, len(embeddings))
quantizer = faiss.IndexFlatIP(dimension)
index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)
index.train(embeddings)
index.add(embeddings)

faiss.write_index(index, str(FAISS_DIR / 'medical_cases.index'))
metadata = {
    'patient_ids': combined_df['patient_id'].tolist(),
    'pathologies': combined_df['pathology'].tolist(),
    'symptoms': combined_df['symptoms_text'].fillna('None reported').tolist(),
    'combined_text': combined_df['combined_text'].fillna('No information available').tolist(),
    'splits': combined_df['source_split'].tolist(),
    'num_vectors': int(index.ntotal),
    'dimension': int(dimension),
    'profile': DATASET_PROFILE,
    'used_full_dataset': DATASET_PROFILE == 'full_dataset',
}
with open(FAISS_DIR / 'metadata_mapping.pkl', 'wb') as handle:
    pickle.dump(metadata, handle)

print({'metadata_keys': sorted(metadata.keys())})
print({'combined_text_sample': metadata['combined_text'][0]})
print({'symptoms_sample': metadata['symptoms'][0]})

with open(FAISS_DIR / 'index_info.json', 'w', encoding='utf-8') as fh:
    json.dump({
        'num_vectors': int(index.ntotal),
        'dimension': int(dimension),
        'profile': DATASET_PROFILE,
        'embed_batch_size': EMBED_BATCH_SIZE,
    }, fh, indent=2)

print({'num_vectors': int(index.ntotal), 'dimension': int(dimension), 'faiss_dir': str(FAISS_DIR)})



## Export Final Artifacts To Drive

In [ ]:
import shutil

dst_classifier = PROFILE_EXPORT_DIR / 'clinicalbert_classifier_natural'
dst_faiss = PROFILE_EXPORT_DIR / 'faiss_data_natural'
dst_processed = PROFILE_EXPORT_DIR / 'processed_ddxplus'

for dst in (dst_classifier, dst_faiss, dst_processed):
    if dst.exists():
        shutil.rmtree(dst)

shutil.copytree(CLASSIFIER_DIR, dst_classifier)
shutil.copytree(FAISS_DIR, dst_faiss)
shutil.copytree(PROCESSED_DIR, dst_processed)

print('Exported classifier to:', dst_classifier)
print('Exported FAISS to:', dst_faiss)
print('Exported processed CSVs to:', dst_processed)
print('Profile export root:', PROFILE_EXPORT_DIR)



## What To Bring Back Here

Copy these folders from Drive into the local project later:

- `clinicalbert_classifier_natural`
- `faiss_data_natural`

Then we will switch the backend config to use them and rerun end-to-end evaluation.

## Optional Track: Targeted Retraining Bundle (for project promotion)

Run this section after finishing the base natural pipeline.

This optional track will:
1. Build targeted_training CSVs from targeted_cases_v1.json
2. Merge targeted rows with base natural train/validate/test CSVs
3. Continue fine-tuning from clinicalbert_classifier_natural
4. Rebuild FAISS for the targeted bundle
5. Export a ready-to-import bundle for this project

Use this when preparing the next promoted bundle for local runtime.

In [ ]:
import csv
import json
import random
from pathlib import Path

TARGETED_CASES_PATH = Path('/content/drive/MyDrive/DDX/targeted/targeted_cases_v1.json')
TARGETED_OUT_DIR = WORK_DIR / 'targeted_training'
TARGETED_OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETED_INCLUDE_FOLLOW_UP = True
TARGETED_SPLIT_SEED = SEED
TARGETED_TRAIN_RATIO = 0.70
TARGETED_VAL_RATIO = 0.15

# Incremental controls
TARGETED_INCREMENTAL_MODE = True
TARGETED_STATE_PATH = EXPORT_ROOT / 'targeted_state' / 'applied_case_ids.json'
TARGETED_STATE_PATH.parent.mkdir(parents=True, exist_ok=True)


def load_targeted_cases(path: Path):
    with path.open('r', encoding='utf-8') as fh:
        payload = json.load(fh)
    if not isinstance(payload, list):
        raise TypeError('targeted_cases JSON must be a list of case objects')
    return payload


def validate_targeted_cases(cases):
    errors = []
    required_fields = ('id', 'raw_text', 'expected_conditions', 'follow_up_answers')
    seen = set()

    for idx, case in enumerate(cases, start=1):
        prefix = f'case[{idx}]'
        if not isinstance(case, dict):
            errors.append(f'{prefix}: case must be a JSON object')
            continue

        for field in required_fields:
            if field not in case:
                errors.append(f"{prefix}: missing required field '{field}'")

        case_id = str(case.get('id', '')).strip()
        if not case_id:
            errors.append(f"{prefix}: 'id' must be non-empty")
        elif case_id in seen:
            errors.append(f"{prefix}: duplicate id '{case_id}'")
        else:
            seen.add(case_id)

        raw_text = str(case.get('raw_text', '')).strip()
        if not raw_text:
            errors.append(f"{prefix}: 'raw_text' must be non-empty")

        expected = case.get('expected_conditions')
        if not isinstance(expected, list) or not expected or not all(str(item).strip() for item in expected):
            errors.append(f"{prefix}: 'expected_conditions' must be a non-empty list of strings")

        follow_up = case.get('follow_up_answers')
        if not isinstance(follow_up, list):
            errors.append(f"{prefix}: 'follow_up_answers' must be a list")

    return errors


def load_applied_case_ids(path: Path):
    if not path.exists():
        return set()
    with path.open('r', encoding='utf-8') as fh:
        payload = json.load(fh)
    if not isinstance(payload, list):
        return set()
    return {str(item).strip() for item in payload if str(item).strip()}


def save_applied_case_ids(path: Path, case_ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as fh:
        json.dump(sorted(case_ids), fh, indent=2, ensure_ascii=False)


def filter_new_cases(cases, applied_case_ids):
    selected = []
    for case in cases:
        case_id = str(case.get('id', '')).strip()
        if not case_id:
            continue
        if case_id in applied_case_ids:
            continue
        selected.append(case)
    return selected


def build_targeted_rows(cases, include_follow_up=True):
    rows = []
    for case in cases:
        case_id = str(case.get('id', '')).strip()
        raw_text = str(case.get('raw_text', '')).strip()
        expected = case.get('expected_conditions') or ['']
        pathology = str(expected[0]).strip()
        language = str(case.get('language', '')).strip() or 'en'
        difficulty = str(case.get('difficulty', '')).strip() or 'hard'
        ambiguity_group = str(case.get('ambiguity_group', '')).strip()

        if not case_id or not raw_text or not pathology:
            continue

        rows.append({
            'patient_id': f'{case_id}_first_turn',
            'pathology': pathology,
            'combined_text': raw_text,
            'symptoms_text': raw_text,
            'language': language,
            'difficulty': difficulty,
            'ambiguity_group': ambiguity_group,
            'case_source': 'targeted_first_turn',
        })

        if include_follow_up:
            answers = [str(item).strip() for item in (case.get('follow_up_answers') or []) if str(item).strip()]
            if answers:
                rows.append({
                    'patient_id': f'{case_id}_clarified',
                    'pathology': pathology,
                    'combined_text': raw_text + '\n\nFollow-up clarification: ' + ' '.join(answers),
                    'symptoms_text': raw_text,
                    'language': language,
                    'difficulty': difficulty,
                    'ambiguity_group': ambiguity_group,
                    'case_source': 'targeted_clarified',
                })

    return rows


def split_rows(rows, seed=42, train_ratio=0.70, val_ratio=0.15):
    shuffled = list(rows)
    random.Random(seed).shuffle(shuffled)

    total = len(shuffled)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    return shuffled[:train_end], shuffled[train_end:val_end], shuffled[val_end:]


def write_csv_rows(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'patient_id',
        'pathology',
        'combined_text',
        'symptoms_text',
        'language',
        'difficulty',
        'ambiguity_group',
        'case_source',
    ]
    with path.open('w', encoding='utf-8', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def read_rows_for_merge(path: Path):
    with path.open('r', encoding='utf-8-sig', newline='') as fh:
        return list(csv.DictReader(fh))


def dedupe_rows(rows):
    seen = set()
    deduped = []
    for row in rows:
        key = ((row.get('combined_text') or '').strip(), (row.get('pathology') or '').strip())
        if not key[0] or not key[1] or key in seen:
            continue
        seen.add(key)
        deduped.append(row)
    return deduped


def write_merged_csv(path: Path, rows):
    if not rows:
        raise ValueError(f'No rows to write for {path}')
    fieldnames = []
    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open('w', encoding='utf-8', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def merge_split(base_csv: Path, targeted_csv: Path, output_csv: Path):
    merged = dedupe_rows(read_rows_for_merge(base_csv) + read_rows_for_merge(targeted_csv))
    write_merged_csv(output_csv, merged)
    return len(merged)


assert TARGETED_CASES_PATH.exists(), f'targeted cases file not found: {TARGETED_CASES_PATH}'

all_cases = load_targeted_cases(TARGETED_CASES_PATH)
errors = validate_targeted_cases(all_cases)
assert not errors, 'targeted cases validation failed:\n' + '\n'.join(errors[:20])

applied_case_ids_before = load_applied_case_ids(TARGETED_STATE_PATH) if TARGETED_INCREMENTAL_MODE else set()
selected_cases = filter_new_cases(all_cases, applied_case_ids_before) if TARGETED_INCREMENTAL_MODE else list(all_cases)
selected_case_ids = {str(case.get('id', '')).strip() for case in selected_cases if str(case.get('id', '')).strip()}

if TARGETED_INCREMENTAL_MODE and not selected_cases:
    TARGETED_BUILD_RESULT = {
        'targeted_cases_total': len(all_cases),
        'targeted_cases_selected': 0,
        'targeted_cases_skipped_as_already_applied': len(applied_case_ids_before),
        'targeted_rows_total': 0,
        'no_new_cases': True,
        'incremental_mode': True,
        'state_path': str(TARGETED_STATE_PATH),
        'targeted_out_dir': str(TARGETED_OUT_DIR),
    }
    print(json.dumps(TARGETED_BUILD_RESULT, indent=2, ensure_ascii=False))
else:
    all_targeted_rows = build_targeted_rows(selected_cases, include_follow_up=TARGETED_INCLUDE_FOLLOW_UP)
    train_targeted, validate_targeted, test_targeted = split_rows(
        all_targeted_rows,
        seed=TARGETED_SPLIT_SEED,
        train_ratio=TARGETED_TRAIN_RATIO,
        val_ratio=TARGETED_VAL_RATIO,
    )

    train_targeted_path = TARGETED_OUT_DIR / 'train_targeted.csv'
    validate_targeted_path = TARGETED_OUT_DIR / 'validate_targeted.csv'
    test_targeted_path = TARGETED_OUT_DIR / 'test_targeted.csv'

    write_csv_rows(train_targeted_path, train_targeted)
    write_csv_rows(validate_targeted_path, validate_targeted)
    write_csv_rows(test_targeted_path, test_targeted)

    train_merged_path = TARGETED_OUT_DIR / 'train_merged.csv'
    validate_merged_path = TARGETED_OUT_DIR / 'validate_merged.csv'
    test_merged_path = TARGETED_OUT_DIR / 'test_merged.csv'

    train_merged_count = merge_split(PROCESSED_DIR / 'train_natural.csv', train_targeted_path, train_merged_path)
    validate_merged_count = merge_split(PROCESSED_DIR / 'validate_natural.csv', validate_targeted_path, validate_merged_path)
    test_merged_count = merge_split(PROCESSED_DIR / 'test_natural.csv', test_targeted_path, test_merged_path)

    applied_case_ids_after = applied_case_ids_before | selected_case_ids
    if TARGETED_INCREMENTAL_MODE:
        save_applied_case_ids(TARGETED_STATE_PATH, applied_case_ids_after)

    TARGETED_BUILD_RESULT = {
        'targeted_cases_total': len(all_cases),
        'targeted_cases_selected': len(selected_cases),
        'targeted_cases_skipped_as_already_applied': len(all_cases) - len(selected_cases),
        'targeted_rows_total': len(all_targeted_rows),
        'targeted_rows': {
            'train': len(train_targeted),
            'validate': len(validate_targeted),
            'test': len(test_targeted),
        },
        'merged_rows': {
            'train': train_merged_count,
            'validate': validate_merged_count,
            'test': test_merged_count,
        },
        'incremental_mode': TARGETED_INCREMENTAL_MODE,
        'state_path': str(TARGETED_STATE_PATH),
        'no_new_cases': False,
        'targeted_out_dir': str(TARGETED_OUT_DIR),
    }
    print(json.dumps(TARGETED_BUILD_RESULT, indent=2, ensure_ascii=False))

### Continue Fine-Tuning Incrementally (No Full Re-Train)

By default this cell runs in incremental mode:
- resumes from the latest available targeted classifier bundle if present
- trains only on newly added targeted cases (not all previous cases)

You can switch back to merged/full behavior by changing `RETRAIN_DATA_MODE` inside the code cell.

Expected input files in incremental mode:
- targeted_training/train_targeted.csv
- targeted_training/validate_targeted.csv
- targeted_training/test_targeted.csv

In [ ]:
import json
import math
import shutil

import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

required_symbols = ['DiagnosisDataset', 'collate_batch', 'move_to_device', 'evaluate_model', 'read_csv_rows']
missing = [name for name in required_symbols if name not in globals()]
assert not missing, 'Run the base classifier training cell first. Missing symbols: ' + ', '.join(missing)

TARGETED_CLASSIFIER_DIR = WORK_DIR / 'clinicalbert_classifier_targeted'
TARGETED_BUILD_RESULT = globals().get('TARGETED_BUILD_RESULT', {})

if TARGETED_BUILD_RESULT.get('no_new_cases'):
    print('No new targeted cases detected in incremental mode. Skipping retrain cell.')
    print(json.dumps(TARGETED_BUILD_RESULT, indent=2, ensure_ascii=False))
else:
    if TARGETED_CLASSIFIER_DIR.exists():
        shutil.rmtree(TARGETED_CLASSIFIER_DIR)
    TARGETED_CLASSIFIER_DIR.mkdir(parents=True, exist_ok=True)

    RETRAIN_TEXT_COLUMN = 'combined_text'
    RETRAIN_LABEL_COLUMN = 'pathology'
    RETRAIN_MAX_LENGTH = MAX_LENGTH
    RETRAIN_BATCH_SIZE = BATCH_SIZE
    RETRAIN_EPOCHS = 2
    RETRAIN_LEARNING_RATE = 1e-5
    RETRAIN_WEIGHT_DECAY = WEIGHT_DECAY

    # 'targeted_only' => incremental on newly-added targeted rows only.
    # 'merged' => use natural+targeted merged rows.
    RETRAIN_DATA_MODE = 'targeted_only'

    TARGETED_RESUME_MODEL_CANDIDATES = [
        PROFILE_EXPORT_DIR / 'targeted_bundle' / 'clinicalbert_classifier_targeted',
        CLASSIFIER_DIR,
    ]

    def resolve_resume_model_dir(candidates):
        for candidate in candidates:
            if (candidate / 'config.json').exists() and (candidate / 'label_map.json').exists():
                return candidate
        raise FileNotFoundError('No valid resume model directory found. Checked: ' + ', '.join(str(path) for path in candidates))

    TARGETED_RESUME_MODEL_DIR = resolve_resume_model_dir(TARGETED_RESUME_MODEL_CANDIDATES)

    if RETRAIN_DATA_MODE == 'targeted_only':
        train_csv = TARGETED_OUT_DIR / 'train_targeted.csv'
        validate_csv = TARGETED_OUT_DIR / 'validate_targeted.csv'
        test_csv = TARGETED_OUT_DIR / 'test_targeted.csv'
    elif RETRAIN_DATA_MODE == 'merged':
        train_csv = TARGETED_OUT_DIR / 'train_merged.csv'
        validate_csv = TARGETED_OUT_DIR / 'validate_merged.csv'
        test_csv = TARGETED_OUT_DIR / 'test_merged.csv'
    else:
        raise ValueError(f'Unsupported RETRAIN_DATA_MODE={RETRAIN_DATA_MODE!r}')

    train_rows = read_csv_rows(train_csv, RETRAIN_TEXT_COLUMN, RETRAIN_LABEL_COLUMN)
    val_rows = read_csv_rows(validate_csv, RETRAIN_TEXT_COLUMN, RETRAIN_LABEL_COLUMN)
    test_rows = read_csv_rows(test_csv, RETRAIN_TEXT_COLUMN, RETRAIN_LABEL_COLUMN)

    assert train_rows, f'{train_csv.name} is empty'
    assert val_rows, f'{validate_csv.name} is empty'
    assert test_rows, f'{test_csv.name} is empty'

    with open(TARGETED_RESUME_MODEL_DIR / 'label_map.json', 'r', encoding='utf-8') as fh:
        base_label_map = json.load(fh)

    base_label2id = {str(k): int(v) for k, v in base_label_map['label2id'].items()}
    base_id2label = {int(k): str(v) for k, v in base_label_map['id2label'].items()}

    merged_labels = {row['label'] for row in train_rows + val_rows + test_rows}
    unseen_labels = sorted(merged_labels - set(base_label2id.keys()))
    assert not unseen_labels, f'Input data contains labels not present in resume model label map: {unseen_labels}'

    retrain_tokenizer = AutoTokenizer.from_pretrained(TARGETED_RESUME_MODEL_DIR)
    retrain_model = AutoModelForSequenceClassification.from_pretrained(TARGETED_RESUME_MODEL_DIR).to(DEVICE)

    train_dataset = DiagnosisDataset(train_rows, retrain_tokenizer, base_label2id, RETRAIN_MAX_LENGTH)
    val_dataset = DiagnosisDataset(val_rows, retrain_tokenizer, base_label2id, RETRAIN_MAX_LENGTH)
    test_dataset = DiagnosisDataset(test_rows, retrain_tokenizer, base_label2id, RETRAIN_MAX_LENGTH)

    train_loader = DataLoader(train_dataset, batch_size=RETRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
    val_loader = DataLoader(val_dataset, batch_size=RETRAIN_BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
    test_loader = DataLoader(test_dataset, batch_size=RETRAIN_BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

    optimizer = AdamW(retrain_model.parameters(), lr=RETRAIN_LEARNING_RATE, weight_decay=RETRAIN_WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * RETRAIN_EPOCHS)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, math.ceil(total_steps * 0.1)),
        num_training_steps=total_steps,
    )

    history_targeted = []
    best_metric_targeted = -1.0
    best_epoch_targeted = 0

    for epoch in range(1, RETRAIN_EPOCHS + 1):
        retrain_model.train()
        train_losses = []

        for batch in train_loader:
            batch = move_to_device(batch, DEVICE)
            optimizer.zero_grad()
            outputs = retrain_model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                labels=batch['labels'],
                token_type_ids=batch.get('token_type_ids'),
            )
            loss = outputs.loss
            loss.backward()
            nn.utils.clip_grad_norm_(retrain_model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_losses.append(float(loss.item()))

        val_metrics = evaluate_model(retrain_model, val_loader, DEVICE, base_id2label)
        epoch_record = {
            'epoch': epoch,
            'train_loss': sum(train_losses) / len(train_losses),
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['averages']['macro_f1'],
        }
        history_targeted.append(epoch_record)
        print(epoch_record)

        if val_metrics['averages']['macro_f1'] > best_metric_targeted:
            best_metric_targeted = val_metrics['averages']['macro_f1']
            best_epoch_targeted = epoch
            retrain_model.save_pretrained(TARGETED_CLASSIFIER_DIR)
            retrain_tokenizer.save_pretrained(TARGETED_CLASSIFIER_DIR)
            with open(TARGETED_CLASSIFIER_DIR / 'label_map.json', 'w', encoding='utf-8') as fh:
                json.dump(
                    {
                        'label2id': base_label2id,
                        'id2label': {str(k): v for k, v in base_id2label.items()},
                    },
                    fh,
                    indent=2,
                )

    best_targeted_model = AutoModelForSequenceClassification.from_pretrained(TARGETED_CLASSIFIER_DIR).to(DEVICE)
    test_metrics_targeted = evaluate_model(best_targeted_model, test_loader, DEVICE, base_id2label)

    with open(TARGETED_CLASSIFIER_DIR / 'history.json', 'w', encoding='utf-8') as fh:
        json.dump(history_targeted, fh, indent=2)

    summary_targeted = {
        'profile': DATASET_PROFILE,
        'mode': 'continue_finetuning_incremental',
        'resume_model_dir': str(TARGETED_RESUME_MODEL_DIR),
        'retrain_data_mode': RETRAIN_DATA_MODE,
        'input_csv': {
            'train': str(train_csv),
            'validate': str(validate_csv),
            'test': str(test_csv),
        },
        'best_epoch': best_epoch_targeted,
        'best_val_macro_f1': best_metric_targeted,
        'test_loss': test_metrics_targeted['loss'],
        'test_accuracy': test_metrics_targeted['accuracy'],
        'test_macro_f1': test_metrics_targeted['averages']['macro_f1'],
        'rows': {
            'train': len(train_rows),
            'validate': len(val_rows),
            'test': len(test_rows),
        },
        'train_params': {
            'epochs': RETRAIN_EPOCHS,
            'batch_size': RETRAIN_BATCH_SIZE,
            'learning_rate': RETRAIN_LEARNING_RATE,
            'max_length': RETRAIN_MAX_LENGTH,
        },
    }

    with open(TARGETED_CLASSIFIER_DIR / 'summary.json', 'w', encoding='utf-8') as fh:
        json.dump(summary_targeted, fh, indent=2)

    pd.DataFrame(test_metrics_targeted['report_rows']).to_csv(
        TARGETED_CLASSIFIER_DIR / 'test_classification_report.csv',
        index=False,
        encoding='utf-8',
    )
    pd.DataFrame(test_metrics_targeted['prediction_rows']).to_csv(
        TARGETED_CLASSIFIER_DIR / 'test_predictions.csv',
        index=False,
        encoding='utf-8',
    )
    pd.DataFrame(
        test_metrics_targeted['confusion'],
        index=[base_id2label[idx] for idx in test_metrics_targeted['labels']],
        columns=[base_id2label[idx] for idx in test_metrics_targeted['labels']],
    ).to_csv(TARGETED_CLASSIFIER_DIR / 'test_confusion_matrix.csv', encoding='utf-8')

    print(json.dumps(summary_targeted, indent=2, ensure_ascii=False))
    print('Saved targeted classifier to:', TARGETED_CLASSIFIER_DIR)

### Rebuild FAISS For Targeted Bundle

This cell rebuilds a FAISS index from merged natural+targeted text so retrieval and classifier stay aligned in the promoted bundle.

In [ ]:
import json
import pickle

import faiss
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

TARGETED_FAISS_DIR = WORK_DIR / 'faiss_data_targeted'
TARGETED_FAISS_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
EMBED_BATCH_SIZE = 32 if DEVICE == 'cuda' else 8

embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
embed_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE)
embed_model.eval()


def mean_pooling_targeted(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)


def encode_texts_targeted(texts, batch_size):
    vectors = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        inputs = embed_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt',
        ).to(DEVICE)

        with torch.no_grad():
            outputs = embed_model(**inputs)

        embeddings = mean_pooling_targeted(outputs, inputs['attention_mask'])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        vectors.append(embeddings.cpu().numpy().astype('float32'))

    return np.vstack(vectors)


merge_frames = []
for split_name in ('train_merged', 'validate_merged'):
    csv_path = TARGETED_OUT_DIR / f'{split_name}.csv'
    assert csv_path.exists(), f'missing merged split for FAISS rebuild: {csv_path}'
    frame = pd.read_csv(csv_path)
    frame['source_split'] = split_name
    merge_frames.append(frame)

faiss_df = pd.concat(merge_frames, ignore_index=True)
faiss_df['combined_text'] = faiss_df['combined_text'].fillna('No information available')
faiss_df['symptoms_text'] = faiss_df.get('symptoms_text', pd.Series(['None reported'] * len(faiss_df))).fillna('None reported')

faiss_texts = faiss_df['combined_text'].tolist()
assert faiss_texts, 'No text rows found for targeted FAISS build'

embeddings = encode_texts_targeted(faiss_texts, EMBED_BATCH_SIZE)
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
nlist = min(100, len(embeddings))
quantizer = faiss.IndexFlatIP(dimension)
index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)
index.train(embeddings)
index.add(embeddings)

faiss.write_index(index, str(TARGETED_FAISS_DIR / 'medical_cases.index'))
metadata = {
    'patient_ids': faiss_df['patient_id'].astype(str).tolist(),
    'pathologies': faiss_df['pathology'].astype(str).tolist(),
    'symptoms': faiss_df['symptoms_text'].astype(str).tolist(),
    'combined_text': faiss_df['combined_text'].astype(str).tolist(),
    'splits': faiss_df['source_split'].astype(str).tolist(),
    'num_vectors': int(index.ntotal),
    'dimension': int(dimension),
    'profile': DATASET_PROFILE,
    'bundle_mode': 'targeted_retrain',
}

with open(TARGETED_FAISS_DIR / 'metadata_mapping.pkl', 'wb') as handle:
    pickle.dump(metadata, handle)

with open(TARGETED_FAISS_DIR / 'index_info.json', 'w', encoding='utf-8') as fh:
    json.dump(
        {
            'num_vectors': int(index.ntotal),
            'dimension': int(dimension),
            'profile': DATASET_PROFILE,
            'embed_batch_size': EMBED_BATCH_SIZE,
            'bundle_mode': 'targeted_retrain',
        },
        fh,
        indent=2,
    )

print(
    json.dumps(
        {
            'faiss_rows': len(faiss_df),
            'num_vectors': int(index.ntotal),
            'dimension': int(dimension),
            'targeted_faiss_dir': str(TARGETED_FAISS_DIR),
        },
        indent=2,
        ensure_ascii=False,
    )
)

### Export Targeted Bundle To Drive

This exports three folders:
- clinicalbert_classifier_targeted
- faiss_data_targeted
- targeted_training

A bundle_manifest.json file is also written for traceability.

In [ ]:
import json
import shutil
from datetime import datetime, timezone

TARGETED_EXPORT_DIR = PROFILE_EXPORT_DIR / 'targeted_bundle'
DST_TARGETED_CLASSIFIER = TARGETED_EXPORT_DIR / 'clinicalbert_classifier_targeted'
DST_TARGETED_FAISS = TARGETED_EXPORT_DIR / 'faiss_data_targeted'
DST_TARGETED_TRAINING = TARGETED_EXPORT_DIR / 'targeted_training'

TARGETED_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for dst in (DST_TARGETED_CLASSIFIER, DST_TARGETED_FAISS, DST_TARGETED_TRAINING):
    if dst.exists():
        shutil.rmtree(dst)

shutil.copytree(TARGETED_CLASSIFIER_DIR, DST_TARGETED_CLASSIFIER)
shutil.copytree(TARGETED_FAISS_DIR, DST_TARGETED_FAISS)
shutil.copytree(TARGETED_OUT_DIR, DST_TARGETED_TRAINING)

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'profile': DATASET_PROFILE,
    'bundle_mode': 'targeted_retrain',
    'bundle_root': str(TARGETED_EXPORT_DIR),
    'artifacts': {
        'classifier': str(DST_TARGETED_CLASSIFIER),
        'faiss': str(DST_TARGETED_FAISS),
        'targeted_training_csv': str(DST_TARGETED_TRAINING),
    },
    'runtime_recommendation': {
        'finetuned_model_dir': 'backend/artifacts/clinicalbert_classifier_targeted',
        'faiss_index_dir': 'backend/artifacts/faiss_data_targeted',
        'rag_top_k': 7,
    },
}

with open(TARGETED_EXPORT_DIR / 'bundle_manifest.json', 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2)

print('Exported targeted classifier to:', DST_TARGETED_CLASSIFIER)
print('Exported targeted FAISS to:', DST_TARGETED_FAISS)
print('Exported targeted CSV package to:', DST_TARGETED_TRAINING)
print('Wrote manifest:', TARGETED_EXPORT_DIR / 'bundle_manifest.json')

## Import Back Into This Project (local step)

After downloading targeted_bundle from Drive to your machine:

1. Copy clinicalbert_classifier_targeted into backend/artifacts/
2. Copy faiss_data_targeted into backend/artifacts/
3. Keep targeted_training as a trace package in project root if needed
4. Update backend environment values:
   - USE_FINETUNED_CLASSIFIER=true
   - FINETUNED_MODEL_DIR=backend/artifacts/clinicalbert_classifier_targeted
   - FAISS_INDEX_DIR=backend/artifacts/faiss_data_targeted
   - RAG_TOP_K=7

Then run the project evaluation scripts locally to confirm release gates on the new bundle.